*0.2 Math / ML basics*

# Break → Fix

**The situation.** A team upgrades the embedding model. To save money they embed only *new* documents with the new model and leave the 2 million existing vectors as they are. Nothing errors. Search quality collapses for old documents, and the dashboard shows nothing wrong because the index still answers every query.

**The bug.** Two models, two spaces. A vector from model A and a vector from model B are lists of the same length, so the dot product runs — but the numbers mean nothing together. Old documents can never match new queries.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The broken version.** Old documents embedded with a local model, the query with OpenAI's. Both 768-and-1536 sized? No — that would error. Use two models with the *same* size so the bug is silent: OpenAI at `dimensions=768` and nomic-embed-text at 768.

In [2]:
import numpy as np
from ollama import Client
from openai import OpenAI

documents = [
    "Duplicate payment on your statement",
    "How to change your profile photo",
    "Two-factor authentication setup",
]
query = "my card was charged twice"

client = OpenAI(timeout=30)
ollama = Client()

# BREAK: documents from one model, query from another — same size, so no error.
old_vectors = np.array(
    ollama.embed(model="nomic-embed-text", input=documents)["embeddings"], dtype=np.float32
)
new_query = np.array(
    client.embeddings.create(model="text-embedding-3-small", input=[query], dimensions=768)
    .data[0]
    .embedding,
    dtype=np.float32,
)
broken_scores = old_vectors @ new_query
print("shapes match:", old_vectors.shape, new_query.shape)
for score, document in sorted(zip(broken_scores, documents), reverse=True):
    print(f"  {score:>6.3f}  {document}")
print("BREAK: scores are all near zero and in no useful order")

shapes match: (3, 768) (768,)
  -0.001  Two-factor authentication setup
  -0.018  Duplicate payment on your statement
  -0.043  How to change your profile photo
BREAK: scores are all near zero and in no useful order


**The fix.** One model per index, and re-embed everything when it changes. Cheap check: record the model name next to the index and refuse queries from any other.

In [3]:
index_model = "text-embedding-3-small@768"


def embed(texts: list[str], model_tag: str) -> np.ndarray:
    assert model_tag == index_model, (
        f"index built with {index_model}, got {model_tag}"
    )  # refuse mixed spaces
    vectors = []
    for item in client.embeddings.create(
        model="text-embedding-3-small", input=texts, dimensions=768
    ).data:
        vectors.append(item.embedding)
    return np.array(vectors, dtype=np.float32)


document_vectors = embed(documents, index_model)  # re-embedded with the same model as the query
query_vector = embed([query], index_model)[0]
fixed_scores = document_vectors @ query_vector
for score, document in sorted(zip(fixed_scores, documents), reverse=True):
    print(f"  {score:>6.3f}  {document}")
print("FIX: the duplicate-payment article is first")
assert documents[int(np.argmax(fixed_scores))] == "Duplicate payment on your statement"
assert fixed_scores.max() > broken_scores.max()

   0.575  Duplicate payment on your statement
   0.268  Two-factor authentication setup
   0.150  How to change your profile photo
FIX: the duplicate-payment article is first


**Reading the output.** Broken: three meaningless scores. Fixed: the right article on top with a clear margin. Same documents, same query — the only change is that both sides now come from one model.

**How you notice it.** Search quality that drops only for older documents; top scores that are lower than they used to be; an index with no record of which model built it.

**Watch out**
- Store the model name and version with the index. Check it on every write and every query.
- Changing `dimensions` on the same model is also a new space.
- A migration means re-embedding everything, or running two indexes side by side until the old one is gone.